# Bivariate bicycle codes

Bivariate bicycle codes (`BBCode`s) are CSS quantum LDPC codes whose check matrices are built from two polynomials in the generators of cyclic groups.
The family includes the codes proposed for the IBM low-overhead quantum-memory experiment in [Nature 627, 778 (2024)](https://www.nature.com/articles/s41586-024-07107-7), also available as [arXiv:2308.07915](https://arxiv.org/abs/2308.07915).

This notebook reconstructs one published code, distinguishes qLDPC's randomized distance bound from the paper's explicit logical-operator witness, and finds equivalent descriptions that make the code's toric-code-like qubit connectivity explicit.
In these "manifestly toric" layouts, qubits and checks are arranged on a periodic two-dimensional lattice, and four of the data qubits connected to each check are neighbors on that lattice; the remaining qubit connections may still be nonlocal.


## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.


In [1]:
%%capture
%pip install qldpc

In [2]:
from sympy.abc import x, y

from qldpc import codes, decoders

## Construct a published code

The code below is the second-to-last entry in Table 3 of [arXiv:2308.07915v2](https://arxiv.org/abs/2308.07915v2).
The paper gives $n=360$ and $k=12$, together with an explicit logical operator proving the upper bound $d\leq24$.
Constructing the code below verifies its length and dimension but does not reconstruct or validate that logical-operator witness.


In [3]:
orders = {x: 30, y: 6}
poly_a = x**9 + y + y**2
poly_b = y**3 + x**25 + x**26
code = codes.BBCode(orders, poly_a, poly_b)

assert (len(code), code.dimension) == (360, 12)
print(code)
print("physical qubits:", len(code))
print("logical qubits:", code.dimension)

BBCode on 360 qubits with cyclic group orders {x: 30, y: 6} and generating polynomials
  A = x**9 + y**2 + y
  B = x**26 + x**25 + y**3
physical qubits: 360
logical qubits: 12


`get_distance_bound` performs a randomized search for a nontrivial logical operator and returns the smallest weight it finds.
That weight is an upper bound on distance, not an exact distance or an independent verification of the published witness.
With only 1000 trials it may return a weaker bound than the published $d\leq24$ witness.


In [4]:
distance_bound = code.get_distance_bound(num_trials=1000, decoder=decoders.bp_lsd())
print("1000-trial randomized upper bound:", distance_bound)
print("published upper bound:", 24)

1000-trial randomized upper bound: 40
published upper bound: 24


## Manifestly toric descriptions

In a manifestly toric description, the generating polynomials begin with $A=1+x+\cdots$ and $B=1+y+\cdots$.
On the corresponding canonical grid, four data-qubit neighbors of every check are local on a torus, although the remaining couplings may still be nonlocal.

`get_equivalent_toric_layout_code_data` returns relabelings of the same code, not new codes with merely matching parameters.


In [5]:
toric_descriptions = code.get_equivalent_toric_layout_code_data()
assert toric_descriptions

for toric_orders, toric_a, toric_b in toric_descriptions:
    print(f"orders={toric_orders}, A={toric_a}, B={toric_b}")

orders=(30, 6), A=x**5*y**3 + x + 1, B=x**9/y + y + 1
orders=(30, 6), A=x + 1 + y**3/x**4, B=y + 1 + 1/(x**9*y)
orders=(30, 6), A=x**5*y**3 + x + 1, B=x**9*y**2 + y + 1
orders=(30, 6), A=x + 1 + y**3/x**4, B=y + 1 + y**2/x**9


## Optimized hardware layouts

The repository also contains [`examples/scripts/find_bbcode_layouts.py`](https://github.com/qLDPCOrg/qLDPC/blob/main/examples/scripts/find_bbcode_layouts.py), which searches the placement strategy from [arXiv:2404.18809](https://arxiv.org/abs/2404.18809).
That script is a research companion, not installed qLDPC API, so run it from a source checkout rather than copying an import that only works from this notebook's directory:

```bash
python examples/scripts/find_bbcode_layouts.py
```

The optimization can be much more expensive than constructing a code or listing its toric relabelings.
